# Bloomberg data extraction

Extracts the daily prices, volumes and GICS metadata used in the paper (Section 4), and writes them in the layout read by `BloombergDataFetcher`:

```bash
data/<INDEX>/daily.csv      # constituents: px_open, px_last, px_low, px_high, volume
data/<INDEX>/index.csv      # the index itself, same fields
data/<INDEX>/metadata.csv   # one row per constituent, incl. gics_sector_name
```

The selection of the 100 most actively traded constituents is not performed here, but at loading time, through `top_n_by_volume`.

**Setup on a Bloomberg Terminal PC:**

1. Create a new virtual environment and install `pip install pandas "xbbg<1.0"`.
2. Install `blpapi` through
```bash
python -m pip install --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi
```
3. Clone this repository and run the notebook from its root, so that `mingle.utils.enums` can be imported.
4. Make sure the Bloomberg Terminal is running and logged in, then modify the parameters that follow as necessary.

*Running this code for the S&P 500 constituents from January 1st 2000 to March 30th 2026 took ~2 minutes.*

In [ ]:
from pathlib import Path

import pandas as pd
from xbbg import blp

from mingle.configs.defaults import START_DATE, END_DATE
from mingle.utils.enums import BloombergField, Index

## Parameters

In [ ]:
# Bloomberg ticker of each index in the universe
INDICES = {
    Index.SPX: "SPX Index",
    Index.NIKKEI: "NKY Index",
    Index.STOXX: "SXXP Index",
}

FIELDS = [field.value for field in BloombergField]

METADATA_FIELDS = [
    "LONG_COMP_NAME", "COUNTRY_FULL_NAME", "CRNCY", "PRIMARY_EXCHANGE_NAME",
    "GICS_SECTOR_NAME", "GICS_INDUSTRY_GROUP_NAME", "GICS_INDUSTRY_NAME", "GICS_SUB_INDUSTRY_NAME",
]

In [2]:
# Prices are converted to a common currency across indices.
CURRENCY = "USD"

DATA_DIR = Path(".")
CHUNK_SIZE = 100 

## Helpers

In [ ]:
def get_members(index: str) -> list[str]:
    """Index constituents on a given date (YYYYMMDD), as Bloomberg equity tickers."""
    members = blp.bds(index, "INDX_MWEIGHT_HIST")
    return [f"{m} Equity" for m in members.iloc[:, 0]]


def get_history(tickers: list[str], fields: list[str], start: str, end: str, currency: str) -> pd.DataFrame:
    """Daily history in the layout read by BloombergDataFetcher.get_prices:
    (Ticker, Field) column MultiIndex with lowercase fields, indexed by date."""
    history = pd.concat(
        [
            blp.bdh(tickers[i:i + CHUNK_SIZE], fields, start, end, Currency=currency)
            for i in range(0, len(tickers), CHUNK_SIZE)
        ],
        axis=1,
    )
    history.index = pd.to_datetime(history.index)
    history.index.name = "date"
    history.columns = pd.MultiIndex.from_tuples(
        [(ticker, field.lower()) for ticker, field in history.columns],
        names=["Ticker", "Field"],
    )
    return history.sort_index()


def get_metadata(tickers: list[str], fields: list[str]) -> pd.DataFrame:
    """Static reference data, one row per ticker, with lowercase column names."""
    metadata = blp.bdp(tickers, fields)
    metadata.columns = metadata.columns.str.lower()
    metadata.index.name = "ticker"
    return metadata.sort_index()

## Extract data for index constituents and the index itself

In [ ]:
for index, ticker in INDICES.items():
    out_dir = DATA_DIR / index.value
    out_dir.mkdir(parents=True, exist_ok=True)

    members = get_members(ticker)

    get_history(members, FIELDS, START_DATE, END_DATE, CURRENCY).to_csv(out_dir / "daily.csv")
    get_history([ticker], FIELDS, START_DATE, END_DATE, CURRENCY).to_csv(out_dir / "index.csv")
    get_metadata(members, METADATA_FIELDS).to_csv(out_dir / "metadata.csv")

## Access pattern

In [ ]:
for index in INDICES:
    daily = pd.read_csv(DATA_DIR / index.value / "daily.csv", header=[0, 1], index_col=0, parse_dates=True)
    close = daily.xs(BloombergField.CLOSE.value, level="Field", axis=1)
    print(f"{index.value}: {close.shape[1]} constituents, {close.notna().all().sum()} with a complete history, "
          f"{close.index.min().date()} to {close.index.max().date()}")